# TabPFN blend (model diversity)
Adds a decorrelated second model to the LightGBM bag. We're at the causal ceiling, so the gain here is **diversity in the blend**, not a stronger single model.

**CPU / 8.6 GB constraints** (no GPU) force three accommodations:
1. TabPFN caps at ~500 features -> use **top-K (~256) by `final_gain_ranking.csv`**.
2. TabPFN context best <=10k rows -> **subsample the training bank** (cap rows/series, keep balance).
3. TabPFN CPU inference is slow per row -> **predict in batches**, subsample the eval grid if needed.

**Fallback:** if TabPFN won't install / OOMs, set `USE_TABPFN=False` to swap in a deliberately decorrelated 2nd GBM (shallow, different subsample) and keep the exact same blend scaffold.

Blend weight is picked on a **held tuning split**, then reported on an untouched split + reduced test.

## 0. Install (run once, slow on CPU)

In [ ]:
# !pip install tabpfn torch --quiet        # torch mac wheel is CPU by default (~2 GB)
USE_TABPFN = True

In [ ]:
import numpy as np, pandas as pd, lightgbm as lgb, time
import sb_common as C, sb_cv as cv
from sb_model import PARAMS, rank01, eq_series_weight, true_step

X, idx, cols = cv.load_bank()
y = cv.load_y()
print('bank', X.shape)

## 1. Top-K features by GBM gain (TabPFN <=500 cap)

In [ ]:
K = 256
gain = pd.read_csv('final_gain_ranking.csv', index_col=0).iloc[:,0]
topk = [c for c in gain.sort_values(ascending=False).index if c in cols][:K]
ci = [cols.index(c) for c in topk]           # integer column subset
print('using', len(ci), 'features')

## 2. Build eval matrices (fixed split, top-K features)
Train context = the fixed `tr_ids` series, subsampled to <=10k rows with class balance. Eval = the fixed holdout, split into a **tune half** (pick blend weight) and a **report half**, plus the independent reduced test.

In [ ]:
tr_ids = np.load('tr_ids.npy')
ids = idx.get_level_values('id').to_numpy()
mtr = np.isin(ids, tr_ids)
yv  = y.reindex(idx).to_numpy().astype(np.int8)

Xk = X[:, ci]
# subsample training context to <=10k rows, balanced
rng = np.random.default_rng(0)
tr_pos = np.where(mtr & (yv==1))[0]; tr_neg = np.where(mtr & (yv==0))[0]
n = min(5000, len(tr_pos), len(tr_neg))
sel = np.concatenate([rng.choice(tr_pos, n, False), rng.choice(tr_neg, n, False)])
Xctx, yctx = Xk[sel], yv[sel]
print('TabPFN context', Xctx.shape, 'pos frac %.2f' % yctx.mean())

In [ ]:
def load_eval(path):
    F = pd.read_parquet(path)[cols]
    Xe = F.to_numpy(np.float32)[:, ci]
    return Xe, y.reindex(F.index).to_numpy().astype(np.int8), true_step(F.index, y), F.index

Xh, yh, th, ih = load_eval('feat_final_holdout.parquet')
Xr, yr, trd, ir = load_eval('feat_final_reduced.parquet')
# split holdout series -> tune half / report half
hu = np.unique(ih.get_level_values('id').to_numpy())
tune_ids = set(np.random.default_rng(1).permutation(hu)[:len(hu)//2].tolist())
hid = ih.get_level_values('id').to_numpy()
mtune = np.isin(hid, list(tune_ids)); mrep = ~mtune
print('holdout tune/report rows', int(mtune.sum()), int(mrep.sum()))

## 3. Diversity model: TabPFN (or decorrelated GBM fallback)

In [ ]:
def batched_proba(clf, Xe, bs=2000):
    out = []
    for i in range(0, len(Xe), bs):
        out.append(clf.predict_proba(Xe[i:i+bs])[:,1])
    return np.concatenate(out)

t0 = time.time()
if USE_TABPFN:
    from tabpfn import TabPFNClassifier
    div = TabPFNClassifier(device='cpu')
    div.fit(Xctx, yctx)
else:
    # deliberately decorrelated 2nd GBM: shallow, low colsample, different seed
    div = lgb.LGBMClassifier(objective='binary', n_estimators=400, learning_rate=0.05,
        num_leaves=15, min_child_samples=200, colsample_bytree=0.4, subsample=0.7,
        subsample_freq=1, reg_lambda=1.0, n_jobs=-1, verbosity=-1, random_state=99)
    div.fit(Xctx, yctx)
dh = batched_proba(div, Xh); dr = batched_proba(div, Xr)
print('diversity model fit+predict %.0fs' % (time.time()-t0))
print('div standalone: holdout-report %.4f | reduced %.4f' % (
    C.ts_auc_arrays(dh[mrep], yh[mrep], th[mrep]),
    C.ts_auc_arrays(dr, yr, trd)))

## 4. GBM predictions (same eval sets) + blend-weight sweep

In [ ]:
# GBM on ALL features (its strength), 3-seed bag
wtr = eq_series_weight(idx[mtr])
gh, gr = [], []
for s in (0,1,2):
    m = lgb.LGBMClassifier(random_state=s, **PARAMS).fit(X[mtr], yv[mtr], sample_weight=wtr)
    gh.append(m.predict_proba(Xh)[:,1]); gr.append(m.predict_proba(Xr)[:,1])
gh = np.mean([rank01(q) for q in gh],0); gr = np.mean([rank01(q) for q in gr],0)
dh_r, dr_r = rank01(dh), rank01(dr)

def blend(a, b, w):  # rank-space convex blend
    return rank01((1-w)*a + w*b)

# pick weight on the TUNE half of holdout
best_w, best = 0.0, -1
for w in np.linspace(0, 0.6, 13):
    sc = C.ts_auc_arrays(blend(gh, dh_r, w)[mtune], yh[mtune], th[mtune])
    if sc > best: best, best_w = sc, w
print('best blend w=%.2f (tune-half %.4f)' % (best_w, best))

## 5. Report on untouched split + reduced test

In [ ]:
g_only = C.ts_auc_arrays(gh[mrep], yh[mrep], th[mrep])
b_rep  = C.ts_auc_arrays(blend(gh, dh_r, best_w)[mrep], yh[mrep], th[mrep])
g_red  = C.ts_auc_arrays(gr, yr, trd)
b_red  = C.ts_auc_arrays(blend(gr, dr_r, best_w), yr, trd)
print('holdout-report  GBM %.4f -> blend %.4f  (%+.4f)' % (g_only, b_rep, b_rep-g_only))
print('reduced test    GBM %.4f -> blend %.4f  (%+.4f)  (base 0.5764)' % (g_red, b_red, b_red-g_red))